# 1. Setup
Run every cell in this section at the start of each Kaggle session (GPU on, internet on).

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = "/root/.local/bin:/usr/local/bin:" + os.environ["PATH"]
os.environ["UV_PROJECT_ENVIRONMENT"] = "/tmp/dqn-venv"
!uv --version

In [ ]:
import os
REPO = "/kaggle/working/Deep-Q-Learning"
if not os.path.exists(REPO):
    !git clone https://github.com/Ebi-Tech/Deep-Q-Learning.git {REPO}
else:
    !git -C {REPO} fetch origin
    !git -C {REPO} reset --hard origin/main
%cd {REPO}
!git log --oneline -1

In [ ]:
!uv sync

In [ ]:
%%bash
uv run python - <<'EOF'
import inspect
import os
import sys

import torch
import train

failures = 0
def check(name, condition):
    global failures
    print("ok    " if condition else "FAILED", name)
    if not condition:
        failures += 1

check(f"current folder is the repo root: {os.getcwd()}",
      os.getcwd() == "/kaggle/working/Deep-Q-Learning" and os.path.isfile("train.py"))
check(f"CUDA is available: {torch.cuda.is_available()}", torch.cuda.is_available())

sys.argv = ["train.py", "--member", "Divine", "--experiment", "0"]
settings = train.get_settings()
check(f"train_every default is 1: {settings.train_every}", settings.train_every == 1)
check(f"total_steps default is 1,000,000: {settings.total_steps:,}", settings.total_steps == 1_000_000)
check("QNetwork uses He (kaiming) initialisation", "kaiming_normal_" in inspect.getsource(train.QNetwork))
check("Adam uses eps=1.5e-4", "eps=1.5e-4" in inspect.getsource(train.main))

print("all checks passed" if failures == 0 else f"{failures} check(s) FAILED: do not start experiments")
EOF

In [ ]:
import csv
import os
import shutil

from IPython.display import FileLink, display

def check_and_download(n):
    """Check that experiment n finished, print its average reward, and show download links for its log and model.

    The log gets a row only when an episode ends, and the unfinished final episode is not logged,
    so the last logged step is always a little under 1,000,000 (by up to one episode length).
    The last-step check is only there to catch runs that were cut off by a Kaggle session ending.
    """
    log_path = f"logs/Divine/experiment_{n}.csv"
    model_path = f"models/Divine/experiment_{n}.pth"
    rows = []
    if os.path.isfile(log_path):
        with open(log_path) as f:
            rows = list(csv.DictReader(f))
    last_step = int(rows[-1]["step"]) if rows else 0
    checks = [
        (f"log {log_path} exists", os.path.isfile(log_path)),
        (f"last step is at least 999,000: {last_step:,}", last_step >= 999_000),
        (f"model {model_path} exists", os.path.isfile(model_path)),
    ]
    for name, passed in checks:
        print("ok    " if passed else "FAILED", name)
    if rows:
        last_100 = [float(row["reward"]) for row in rows[-100:]]
        print(f"average reward over the last {len(last_100)} episodes: {sum(last_100) / len(last_100):.2f}")

    copies = []
    for path in (log_path, model_path):
        if os.path.isfile(path):
            name = "Divine_" + os.path.basename(path)
            shutil.copy(path, os.path.join("/kaggle/working", name))
            copies.append(name)
    repo = os.getcwd()
    os.chdir("/kaggle/working")
    try:
        for name in copies:
            display(FileLink(name))
    finally:
        os.chdir(repo)
    print("These files are also in Kaggle's Output panel under /kaggle/working.")

# 2. Experiments
One cell per experiment, so they can be split across Kaggle sessions. Each run uses the defaults in train.py except the setting named above it. Never rerun a finished experiment: it restarts training and overwrites its log.

Exp 1: baseline, all default settings.

In [ ]:
!uv run train.py --member Divine --experiment 1
check_and_download(1)

Exp 2: learning rate 5e-5 (lower than baseline).

In [ ]:
!uv run train.py --member Divine --experiment 2 --learning_rate 5e-5
check_and_download(2)

Exp 3: learning rate 2.5e-4 (higher than baseline).

In [ ]:
!uv run train.py --member Divine --experiment 3 --learning_rate 2.5e-4
check_and_download(3)

Exp 4: learning rate 5e-4.

In [ ]:
!uv run train.py --member Divine --experiment 4 --learning_rate 5e-4
check_and_download(4)

Exp 5: learning rate 1e-3.

In [ ]:
!uv run train.py --member Divine --experiment 5 --learning_rate 1e-3
check_and_download(5)

Exp 6: gamma 0.95 (less weight on future rewards).

In [ ]:
!uv run train.py --member Divine --experiment 6 --gamma 0.95
check_and_download(6)

Exp 7: gamma 0.9.

In [ ]:
!uv run train.py --member Divine --experiment 7 --gamma 0.9
check_and_download(7)

Exp 8: gamma 0.999 (more weight on future rewards).

In [ ]:
!uv run train.py --member Divine --experiment 8 --gamma 0.999
check_and_download(8)

Exp 9: best learning rate + best gamma. Exp 10: second best learning rate + best gamma. Fill these in after experiments 2 to 8.

In [ ]:
# !uv run train.py --member Divine --experiment 9 --learning_rate BEST_LR --gamma BEST_GAMMA
# check_and_download(9)

In [ ]:
# !uv run train.py --member Divine --experiment 10 --learning_rate SECOND_BEST_LR --gamma BEST_GAMMA
# check_and_download(10)

# 3. Save everything
Backup: zip all logs and models from this session into one file.

In [ ]:
!cd /kaggle/working/Deep-Q-Learning && zip -r /kaggle/working/Divine_results.zip logs/Divine models/Divine
os.chdir("/kaggle/working")
display(FileLink("Divine_results.zip"))
os.chdir("/kaggle/working/Deep-Q-Learning")